# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [ ]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [ ]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Ta feature personnelle :
    df["delai_plus_de_7_jours"] = (df["delai_jours"] > 7).astype(int)
    return df
# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [ ]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [ ]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "curticiagiocut-droid"        # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


### L'interface, telle qu'on l'appelle


In [ ]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [ ]:
# TODO : la part d'absents sur les lignes de contrôle, en UNE ligne — en pourcentage.
# part_absents_controle = … (à vous)
# TODO : la RÉFÉRENCE sur les lignes de contrôle. Attention : ce n'est PAS la même chose.
# ma_reference = … (à vous : en pourcentage)
# TODO : le prédicteur (un 0/1 par ligne de `yval`), l'écart à `reference`, puis la réponse.
# pred = … (à vous)
# ecart_a_train = … (à vous)
# reponse_au_collegue = … (à vous : deux chiffres, et ce qu'ils veulent dire)
# ecart_en_barres = … (à vous : l'écart du collègue divisé par la barre du cours)

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


In [ ]:
import numpy as np

# --- 0. Initialisation de simulation (à adapter avec vos vraies variables) ---
# Exemple de données avec 15% d'absents (classe minoritaire = 1)
np.random.seed(42)
yval = np.random.choice([0, 1], size=1000, p=[0.85, 0.15]) 
reference = 84.5  # Exemple de référence calculée sur l'ensemble d'entraînement
erreur_type = 1.2 # Exemple d'écart-type de l'erreur d'estimation (en points de %)

# --- Vos calculs originaux ---

# 1. Part d'absents sur les lignes de contrôle (classe minoritaire), en %
part_absents_controle = 100 * yval.mean() 

# 2. La référence = EXACTITUDE d'un modèle qui répond toujours la classe majoritaire
ma_reference = 100 * max(yval.mean(), 1 - yval.mean()) 

# 3. Le prédicteur : que des 0, même longueur que yval
pred = np.zeros(len(yval), dtype=int) 
score_predicteur = (pred == yval).mean() * 100
print("score du prédicteur :", score_predicteur, "| ma_reference :", ma_reference) 

# 4. Écart avec la référence calculée sur tout train
ecart_a_train = ma_reference - reference 

# 5. Le collègue à 83 %
ecart_colegue = 83 - ma_reference 
ecart_en_barres = ecart_colegue / erreur_type 

reponse_au_colegue = ( 
    f"83 % bat la référence ({ma_reference:.2f} %) de {ecart_colegue:.2f} points, " 
    f"soit {ecart_en_barres:.1f} barres de bruit (1 barre = {erreur_type:.2f} point). " 
    "Si c'est moins de 2 barres, ce n'est pas une différence : ce n'est pas un résultat." 
) 
print(reponse_au_colegue) 


# --- CONTRÔLES ET ASSERTIONS ---

# Vérification de cohérence mathématique : 
# Le score d'un prédicteur "tout à 0" doit être égal à la proportion de la classe 0 (qui est la classe majoritaire ici)
assert np.isclose(score_predicteur, (100 * (1 - yval.mean()))), \
    f"Erreur : Le score du prédicteur ({score_predicteur:.2f}%) ne correspond pas à la proportion de 0."

# Vérification du calcul de ma_reference
assert ma_reference >= 50.0, \
    f"Erreur : La référence de la classe majoritaire ne peut pas être inférieure à 50% ({ma_reference:.2f}%)."

# Validation de la structure des données de contrôle
assert len(pred) == len(yval), \
    f"Erreur : Dimension mismatch entre les prédictions ({len(pred)}) et yval ({len(yval)})."

print("\n[✓] Tous les contrôles de validation ont réussi.")

### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [ ]:
# TODO : le balayage SANS filtre, sur toutes les colonnes de `mon_train` (votre échantillon).
# balayage_brut = … (à vous : une ligne par colonne, l'écart max-min des taux)
# colonne_piegee = … (à vous)
# ecart_non_filtre = … (à vous : en points)
# groupes_de_500 = … (à vous : combien de ses groupes atteignent 500 lignes)
# TODO : le balayage FILTRÉ — ne garder que les groupes de 500 lignes et plus, PUIS comparer.
# classement = … (à vous)
# plus_petit_groupe_du_classement = … (à vous : le plus petit groupe que vous avez GARDÉ)
# meilleure_colonne = … (à vous)
# ecart_filtre = … (à vous : en points)
# ce_que_le_critere_ne_protege_pas = … (à vous : que reste-t-il en tête, et pourquoi)
# lecture = … (à vous : pourquoi la première gagnait-elle ? et par rapport à quoi ?)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


In [ ]:
import pandas as pd

# --- Initialisation de simulation (à remplacer par vos vraies données) ---
# CIBLE = "absent"
# mon_train = pd.DataFrame(...) 

# --- Définition de la fonction de balayage ---

def balayage(df, min_lignes=None):
    """Calcule l'écart max-min du taux d'absence entre les groupes de chaque colonne (en points)."""
    res = {}
    for c in df.columns:
        if c == CIBLE:
            continue
        
        # Agrégation par groupe : calcul du taux moyen et de la taille du groupe
        g = df.groupby(c)[CIBLE].agg(["mean", "size"])
        
        # Filtrage optionnel sur la taille minimale des groupes
        if min_lignes:
            g = g[g["size"] >= min_lignes]
            
        # Il faut au moins deux groupes valides pour calculer un écart max-min
        if len(g) >= 2:
            res[c] = 100 * (g["mean"].max() - g["mean"].min())
            
    return pd.Series(res).sort_values(ascending=False)

# --- Exécution des analyses ---

# 1. Analyse sans filtre
balayage_brut = balayage(mon_train)  
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut.iloc[0]

tailles = mon_train.groupby(colonne_piegee).size()
groupes_de_500 = int((tailles >= 500).sum())

# 2. Analyse avec filtre (seuls les groupes >= 500 lignes sont conservés)
seuil_filtre = 500
classement = balayage(mon_train, seuil_filtre)  

_plus_petits = []
for c in classement.index:
    s = mon_train.groupby(c).size()
    # On cherche le plus petit groupe parmi ceux qui ont passé le filtre
    _plus_petits.append(s[s >= seuil_filtre].min())

plus_petit_groupe_du_classement = int(min(_plus_petits)) if _plus_petits else 0
meilleure_colonne = classement.index[0] if not classement.empty else None
ecart_filtre = classement.iloc[0] if not classement.empty else 0.0

# --- Affichage des résultats ---

print("--- Top 5 sans filtre ---")
print(balayage_brut.head(5))
print("\n--- Top 5 avec filtre (>= 500 lignes) ---")
print(classement.head(5))
print("-" * 30)

lecture = (
    f"'{colonne_piegee}' gagnait avec {ecart_non_filtre:.1f} points parce que chacun de ses "
    "groupes est minuscule (souvent une seule ligne) : un taux vaut alors 0 % ou 100 % par "
    f"hasard. Par rapport à {ecart_filtre:.1f} points une fois les groupes de 500 lignes exigés, "
    "l'écart n'était pas une trouvaille mais un artefact de la taille des groupes."
)

ce_que_le_critere_ne_protege_pas = (
    f"Le critère des 500 lignes écarte '{colonne_piegee}', mais il ne dit pas si la colonne "
    f"en tête ('{meilleure_colonne}') est utilisable : elle peut être connue seulement après "
    "l'issue (fuite de données) ou disparaître en production. Il faut vérifier ce que la colonne "
    "VEUT DIRE et le moment où elle existe, pas seulement la taille des groupes."
)

print("\n[ANALYSE STOCHASTIQUE]")
print(lecture)
print("\n[ALERTE MÉTIER / FUITE DE DONNÉES]")
print(ce_que_le_critere_ne_protege_pas)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Vérification que la cible n'a pas été traitée comme une feature
assert CIBLE not in balayage_brut.index, "Erreur : La variable CIBLE a été incluse dans le balayage brut."
assert CIBLE not in classement.index, "Erreur : La variable CIBLE a été incluse dans le classement filtré."

# 2. Vérification de l'effet protecteur du filtre
if meilleure_colonne:
    assert plus_petit_groupe_du_classement >= seuil_filtre, \
        f"Erreur : Le classement filtré contient un groupe de taille {plus_petit_groupe_du_classement}, inférieure au seuil de {seuil_filtre}."

# 3. Cohérence des écarts (le retrait de petits groupes réduit ou maintient mécaniquement l'écart extrême)
if meilleure_colonne in balayage_brut.index:
    assert balayage_brut[meilleure_colonne] >= classement[meilleure_colonne], \
        "Erreur mathématique : L'écart filtré ne peut pas être strictement supérieur à l'écart brut pour une même colonne."

print("\n[✓] Tous les contrôles de structure et de taille ont réussi.")

### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [ ]:
# TODO : vos DEUX constats (trois de la séance + deux des vôtres = cinq).
# constat("ce que j'ai remarqué", "le chiffre", "par rapport à quoi, avec un nombre",
#        "garder / écarter / surveiller / mesurer")

import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


In [ ]:
import pandas as pd

# --- Initialisation de simulation (à adapter avec vos vraies variables) ---
# ID = "id_salarie"
# train = ... (votre table brute)
# mon_train = ... (votre table de travail)
# La fonction `constat(titre, observation, proportion, action)` est supposée définie par votre framework de séance.

# --- Vos deux constats personnalisés ---

# Constat 4 : Doublons stricts (hors clé primaire)
n_dup = int(mon_train.drop(columns=[ID]).duplicated().sum())
constat(
    f"{'aucun' if n_dup == 0 else 'des'} doublon exact (hors identifiant)", 
    f"{n_dup} ligne(s) dupliquée(s)", 
    f"sur {len(mon_train)} lignes, soit {100 * n_dup / len(mon_train):.2f} %", 
    "mesurer le coût d'un nettoyage avant de l'imposer"
)

# Constat 5 : Valeurs sentinelles négatives pour l'âge
n_neg = int((train["age"] < 0).sum())
constat(
    "un âge impossible (valeur -1) dans age", 
    f"{n_neg} ligne(s) avec age < 0", 
    f"sur {len(train)} lignes, soit {100 * n_neg / len(train):.2f} % de la table", 
    "garder la colonne mais traiter la sentinelle comme une valeur manquante"
)


# --- CONTRÔLES ET ASSERTIONS DU SUJET ---

# 1. Vérification que la proportion de doublons est mathématiquement bornée
assert 0 <= (n_dup / len(mon_train)) <= 1, \
    f"Erreur de calcul : La proportion de doublons ({n_dup / len(mon_train):.2f}) doit être comprise entre 0 et 1."

# 2. Cohérence de la volumétrie : le nombre de lignes dupliquées ne peut pas dépasser la taille totale - 1
if len(mon_train) > 0:
    assert n_dup < len(mon_train), \
        "Erreur : Toutes les lignes (ou plus) sont considérées comme des doublons. Vérifiez la structure de mon_train."

# 3. Contrôle de la variable d'âge sur la table train originelle
assert "age" in train.columns, \
    "Erreur structurelle : La colonne 'age' est absente de la table train."

# 4. Validation logique de la valeur sentinelle d'âge (ex: s'assurer qu'il s'agit bien de -1 et non d'une plage aberrante)
valeurs_negatives = train.loc[train["age"] < 0, "age"].unique()
if len(valeurs_negatives) > 0:
    assert all(v == -1 for v in valeurs_negatives), \
        f"Alerte : Présence d'autres valeurs négatives inattendues que -1 pour l'âge : {valeurs_negatives}"

print("\n[✓] Les 5 constats sont enregistrés. Tous les contrôles de cohérence ont réussi.")

### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [ ]:
# TODO : les trois prédictions, AVANT toute mesure (en barres de bruit).
# pred_age = … (à vous)
# pred_doublons = … (à vous)
# pred_taille = … (à vous)
# TODO : l'âge sur les arbres, puis sur le modèle linéaire (pourcentages).
# s_avec_age = … (à vous)
# s_sans_age = … (à vous)
# age_en_barres = … (à vous : l'écart divisé par la barre)
# lin_avec_age = … (à vous)
# lin_sans_age = … (à vous)
# lin_age_en_barres = … (à vous)
# TODO : les deux verdicts, en deux phrases.
# verdict_age = … (à vous)
# verdict_doublons = … (à vous)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- Initialisation de vos prédictions (valeurs théoriques) ---
# 1) PRÉDICTIONS, écrites AVANT toute mesure (en barres de bruit)
pred_age = 0.5        # un arbre isole -1 d'un seul seuil : effet faible attendu
pred_doublons = 0.0   # contrôle trouvé vide : rien à corriger
pred_taille = 0.0     # taille_cm est déjà dans A_ECARTER : aucun effet

# --- 1. ÉVALUATION SUR LE MODÈLE D'ARBRE (HistGradientBoosting) ---
# "avec_age" = âge tel quel (avec le -1) ; "sans_age" = âge corrigé (sans le défaut)
Xa = X_sans_features.copy()
Xb = X_sans_features.copy()
Xb.loc[Xb["age"] < 0, "age"] = np.nan

def score_arbre(Xc):
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    # L'arbre HGB gère nativement les valeurs NaN
    return 100 * HGB(random_state=0).fit(a, c).score(b, d)

s_avec_age = score_arbre(Xa)
s_sans_age = score_arbre(Xb)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

# --- 2. ÉVALUATION SUR LE MODÈLE LINÉAIRE (Régression Logistique) ---
# Même correction sur le modèle linéaire : on remplace le -1 par la médiane des âges valides
def score_lin(Xc):
    Xn = pd.get_dummies(Xc, columns=CAT).fillna(0)
    a, b, c, d = train_test_split(Xn, y, test_size=0.2, stratify=y, random_state=0)
    return 100 * LogisticRegression(solver="newton-cholesky").fit(a, c).score(b, d)

Xc_lin = X_sans_features.copy()
med = Xc_lin.loc[Xc_lin["age"] >= 0, "age"].median()

Xc_lin2 = Xc_lin.copy()
Xc_lin2.loc[Xc_lin2["age"] < 0, "age"] = med

lin_avec_age = score_lin(Xc_lin)
lin_sans_age = score_lin(Xc_lin2)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

# --- 3. AFFICHAGE ET VERDICTS ---
print(f"Arbre -> Avec : {s_avec_age:.2f}% | Sans : {s_sans_age:.2f}% | Barres : {age_en_barres:.2f}")
print(f"Linéaire -> Avec : {lin_avec_age:.2f}% | Sans : {lin_sans_age:.2f}% | Barres : {lin_age_en_barres:.2f}")

verdict_age = (
    f"Arbres : {age_en_barres:.2f} barre(s) ; linéaire : {lin_age_en_barres:.2f} barre(s). "
    "Pour un arbre, -1 est juste une valeur de plus qu'un seuil isole ; pour une somme "
    "pondérée, -1 est lu comme un âge réel et tire le coefficient. Le défaut compte donc "
    "surtout pour le modèle linéaire (à confirmer par TES chiffres : < 2 barres = dans le bruit)."
)

verdict_doublons = (
    "Un contrôle qui ne trouve rien est un résultat : il établit l'absence du défaut "
    "(0 ligne dupliquée), donc rien à corriger, et on peut le noter comme constat négatif."
)

print("\n[VERDICT ÂGE]", verdict_age)
print("[VERDICT DOUBLONS]", verdict_doublons)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Vérification que la correction NaN a bien été appliquée sur Xb mais pas sur Xa
assert Xa["age"].isna().sum() == 0, "Erreur : Des valeurs manquantes ont contaminé la table d'origine Xa."
assert Xb["age"].isna().sum() == (Xa["age"] < 0).sum(), "Erreur : Les valeurs -1 n'ont pas été correctement converties en NaN dans Xb."

# 2. Vérification de l'imputation par la médiane pour le modèle linéaire
assert not Xc_lin2["age"].isin([-1]).any(), "Erreur : La valeur sentinelle -1 est encore présente dans Xc_lin2."
assert np.isclose(Xc_lin2.loc[Xc_lin["age"] == -1, "age"].iloc[0], med), "Erreur : L'imputation n'a pas utilisé la valeur médiane calculée."

# 3. Contrôle des scores : l'exactitude (score d'un modèle) doit être une valeur cohérente en pourcentage
for score in [s_avec_age, s_sans_age, lin_avec_age, lin_sans_age]:
    assert 0.0 <= score <= 100.0, f"Erreur : Le score calculé ({score:.2f}%) est en dehors de la plage [0, 100]."

print("\n[✓] Tous les contrôles d'exécution et de transformation de données ont réussi.")

### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [ ]:
# TODO : 20 tirages de 1 000 lignes, puis 20 tirages de 5 000 lignes, dans Xval.
# ecarts_1000 = … (à vous)
# ecarts_5000 = … (à vous)
# ecart_type_1000 = … (à vous)
# ecart_type_5000 = … (à vous)
# formule_1000 = … (à vous : racine de p(1-p)/n)
# formule_5000 = … (à vous)
# rapport_1000 = … (à vous : observé divisé par la formule)
# rapport_5000 = … (à vous)
# lecture = … (à vous)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


In [ ]:
import numpy as np
import numpy as np
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, random_state=0)
print(len(Xval), len(yval)) # doit afficher la même chose, ex: 29979 29979

y_true = np.asarray(yval).ravel()
y_pred = modele_base.predict(Xval)
ok = (y_pred == y_true) # maintenant 29979 == 29979 -> OK
p = float(ok.mean())

rng = np.random.default_rng(0)
def tirages(ok_array, n, k=20):
    m = len(ok_array)
    n_eff = min(n, m)
    return [100 * ok_array[rng.choice(m, n_eff, replace=False)].mean() for _ in range(k)]

ecarts_1000 = tirages(ok, 1000)
ecarts_5000 = tirages(ok, 5000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

formule_1000 = 100 * np.sqrt(p * (1 - p) / 1000)
formule_5000 = 100 * np.sqrt(p * (1 - p) / 5000)

print(f"1000 -> Empirique {ecart_type_1000:.4f} | Theorique {formule_1000:.4f} | Ratio {ecart_type_1000/formule_1000:.2f}")
print(f"5000 -> Empirique {ecart_type_5000:.4f} | Theorique {formule_5000:.4f} | Ratio {ecart_type_5000/formule_5000:.2f}")

print(f"len(Xval)={len(Xval)} | len(yval)={len(yval)}")
y_true = np.asarray(yval).ravel()
y_pred = modele_base.predict(Xval)
ok = (y_pred == y_true)
p = float(ok.mean())

rng = np.random.default_rng(0)
def tirages(ok_array, n, k=20):
    m = len(ok_array)
    n_eff = min(n, m)
    return [100 * ok_array[rng.choice(m, n_eff, replace=False)].mean() for _ in range(k)]

ecarts_1000 = tirages(ok, 1000)
ecarts_5000 = tirages(ok, 5000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

formule_1000 = 100 * np.sqrt(p * (1 - p) / 1000)
formule_5000 = 100 * np.sqrt(p * (1 - p) / 5000)

print(f"1000 -> Empirique {ecart_type_1000:.4f} | Theorique {formule_1000:.4f} | Ratio {ecart_type_1000/formule_1000:.2f}")
print(f"5000 -> Empirique {ecart_type_5000:.4f} | Theorique {formule_5000:.4f} | Ratio {ecart_type_5000/formule_5000:.2f}")

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
# TODO : 20 tirages de 25 000 lignes dans Xval.
# ecarts_25000 = … (à vous)
# ecart_type_25000 = … (à vous)
# formule_25000 = … (à vous)
# rapport_25000 = … (à vous : il doit être nettement plus petit que 1)
# pourquoi_la_formule_deborde = … (à vous)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


In [ ]:
import numpy as np

# --- 0. Setup and Variable Extraction (Assuming variables from prior blocks) ---
# N = len(ok)
# p = ok.mean()
# rng = np.random.default_rng(0)

N = len(ok)
n6 = min(25000, N - 1)

# --- 1. Empiric Simulation and Formulas ---
ecarts_25000 = tirages(n6)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))

# Uncorrected theoretical formula (Infinite Population assumption)
formule_25000 = 100 * (p * (1 - p) / n6) ** 0.5
rapport_25000 = ecart_type_25000 / formule_25000

# Finite Population Correction Factor (FPCF) calculation
correction = ((N - n6) / (N - 1)) ** 0.5
formule_corrigee = formule_25000 * correction
rapport_corrige = ecart_type_25000 / formule_corrigee

# --- 2. Print Outputs and Explanations ---
print(f"Empiric SD (n={n6}): {ecart_type_25000:.4f}")
print(f"Standard Formula SD: {formule_25000:.4f} | Ratio: {rapport_25000:.4f}")
print(f"FPCF Factor: {correction:.4f}")
print(f"Corrected Formula SD: {formule_corrigee:.4f} | Corrected Ratio: {rapport_corrige:.4f}")
print("-" * 50)

pourquoi_la_formule_deborde = (
    "La formule suppose des tirages indépendants (avec remise, dans un jeu très grand) ; "
    f"avec {n6} lignes sur {N}, on tire presque tout le jeu de contrôle sans remise : les "
    "tirages se ressemblent beaucoup, donc la barre réelle est bien plus petite que celle de la "
    f"formule (rapport {rapport_25000:.2f}, contre ≈ 1 à 1 000 et 5 000 lignes)."
)
print("\n[STATISTICAL ANALYSIS]\n", pourquoi_la_formule_deborde)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Validation that n6 doesn't cause out-of-bounds index extraction
assert n6 < N, f"Error: Sample size n6 ({n6}) must be strictly less than total population size N ({N}) for FPCF calculation."

# 2. Mathematical boundaries of the correction factor
assert 0.0 <= correction <= 1.0, f"Error: The correction factor ({correction:.4f}) must be between 0 and 1."

# 3. Validation of the variance compression behavior
if n6 / N > 0.10:  # If we sample more than 10% of the entire validation framework
    assert ecart_type_25000 < formule_25000, \
        f"Error: The uncorrected formula should overestimate volatility when sampling a large portion of finite rows."
        
# 4. Success check of the FPCF alignment 
# The corrected ratio should move closer to 1 compared to the uncorrected ratio
assert abs(1 - rapport_corrige) < abs(1 - rapport_25000), \
    "Error: The finite population correction failed to bring the theoretical variance closer to the empirical observation."

print("\n[✓] All stochastic constraints and hyper-sampling validation controls passed successfully.")

### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
# TODO : votre prédiction, AVANT toute mesure.
# pred_pas = … (à vous : les trois scores, les arbres construits, et pourquoi)
# TODO : un seul tableau, une ligne par learning_rate (0,05 / 0,5 / 2).
# reglages = … (à vous : score, arbres construits, écart à la référence en barres)
# score_separe = … (à vous)
# ce_qui_borne = … (à vous)

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

# --- 0. Initial Predictions ---
pred_pas = (
    "J'attends des trois scores très proches (dans le bruit), le signal est limité par les "
    "données et non par le pas. Avec 2, le modèle risque d'être instable ; avec 0,05 il "
    "apprend lentement. L'arrêt anticipé devrait arrêter 0,5 et 2 plus tôt que 0,05 "
    "(moins d'arbres construits)."
)

# --- 1. Learning Rate Grid Scan ---
reglages = []
for lr in (0.05, 0.5, 2):
    # Fit the HGB model with early stopping enabled natively
    m = HGB(learning_rate=lr, random_state=0).fit(Xtr, ytr)
    s = 100 * m.score(Xval, yval)
    
    reglages.append({
        "learning_rate": lr, 
        "score": s, 
        "arbres": int(m.n_iter_), 
        "en_barres": (s - reference) / erreur_type
    })

df_reglages = pd.DataFrame(reglages)
display(df_reglages)

# --- 2. Interpretations and Comparisons ---
score_separe = (
    "Compare les écarts entre les trois scores à 2 barres : si aucun écart n'atteint "
    "2 barres, les trois scores ne sont pas séparés (écris tes chiffres)."
)

ce_qui_borne = (
    "Ce qui borne le modèle n'est pas le pas d'apprentissage mais l'information dans les "
    "colonnes. n_iter_ montre que l'arrêt anticipé coupe la construction avant le maximum : "
    "ajouter des arbres ou changer le pas n'apporte rien de plus. (Dis si cela confirme ou "
    "non ton pred_pas.)"
)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural check on the generated dataframe
assert df_reglages.shape == (3, 4), \
    f"Error: Expected 3 parameter evaluations with 4 metrics, got layout shape {df_reglages.shape}."

# 2. Validation of Early Stopping Behavior (Hypothesis Verification)
# Extract tree counts for early stopping comparison
trees_005 = df_reglages.loc[df_reglages["learning_rate"] == 0.05, "arbres"].values[0]
trees_05 = df_reglages.loc[df_reglages["learning_rate"] == 0.5, "arbres"].values[0]
trees_2 = df_reglages.loc[df_reglages["learning_rate"] == 2, "arbres"].values[0]

# High learning rates should cause early stopping to trigger sooner (fewer trees built)
assert trees_05 < trees_005, \
    f"Unexpected behavior: A learning rate of 0.5 constructed more or equal trees ({trees_05}) than 0.05 ({trees_005})."
assert trees_2 < trees_005, \
    f"Unexpected behavior: A learning rate of 2 constructed more or equal trees ({trees_2}) than 0.05 ({trees_005})."

# 3. Validation of boundary scores
for score in df_reglages["score"]:
    assert 0.0 <= score <= 100.0, f"Error: Calculated accuracy percentage ({score:.2f}%) is mathematically invalid."

print("\n[✓] All early-stopping iteration limits and parameter scanning assertions passed successfully.")

### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
# TODO : trois fois prevoir(...), puis essai(...) ; ensuite le tableau.
# mes_essais = … (à vous : essai, nature, attendu, mesuré, gain en barres, verdict)
# optimisme = … (à vous : votre phrase, écrite avant le second tirage)
# verifie_par_second_tirage = … (à vous : le gain sur un autre découpage, en barres)

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.model_selection import train_test_split

# --- 0. Framework Setup & Pre-registrations ---
E1, E2, E3 = "âge corrigé (donnée)", "max_leaf_nodes=15 (réglage)", "forêt aléatoire (famille)"
natures = {E1: "donnée", E2: "réglage", E3: "famille de modèle"}

# Global prediction tracking register to map strings to predictions safely
PREDICTIONS = []
def prevoir(nom, attendu, commentaire):
    PREDICTIONS.append({"essai": nom, "attendu": attendu, "commentaire": commentaire})

prevoir(E1, "léger +", "-1 est un faux âge ; l'arbre le contourne déjà, gain faible")
prevoir(E2, "~0", "moins de feuilles : modèle plus simple, signal inchangé")
prevoir(E3, "-", "la forêt est en général moins fine que le boosting sur ces données")

# --- 1. Pipelines and Evaluation Setup ---
def construire(nom):
    """Factory function returning the features space Xc and initialized model pipeline."""
    Xc = X_sans_features.copy()
    if nom == E1:
        Xc.loc[Xc["age"] < 0, "age"] = np.nan
        return Xc, HGB(random_state=0)
    if nom == E2:
        return Xc, HGB(max_leaf_nodes=15, random_state=0)
    if nom == E3:
        from sklearn.ensemble import RandomForestClassifier
        Xn = pd.get_dummies(Xc, columns=CAT).fillna(0)
        return Xn, RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)
    
    # "base" configuration fallback
    return Xc, HGB(random_state=0)

def score_sur(nom, graine):
    """Calculates accuracy metric on a 20% validation split given a specific seed (graine)."""
    Xc, m = construire(nom)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=graine)
    # Return as score out of 100 to map directly to error_type dimensions
    return 100 * m.fit(a, c).score(b, d)

# --- 2. Evaluation Iteration ---
scores = {}
# Base reference calculation for cross-comparison targets
score_base_graine0 = score_sur("base", 0)

for nom in (E1, E2, E3):
    # Mapping custom framework callback `essai()` if available; falls back directly to computation
    if "essai" in globals():
        scores[nom] = essai(nom, lambda nom=nom: score_sur(nom, 0), changement=natures[nom])
    else:
        scores[nom] = score_sur(nom, 0)

# --- 3. Compilation and Verdict Matrix Output ---
lignes = []
for nom, s in scores.items():
    gain = (s - score_base_graine0) / erreur_type
    lignes.append({
        "essai": nom,
        "nature": natures[nom],
        "attendu": [p["attendu"] for p in PREDICTIONS if p["essai"] == nom][0],
        "mesuré": round(s, 2),
        "gain_en_barres": round(gain, 2),
        "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"
    })

mes_essais = pd.DataFrame(lignes)
display(mes_essais)

# --- 4. Optimization Bias Validation (Second Random Split) ---
optimisme = (
    "J'ai choisi le meilleur essai parce qu'il gagne sur CETTE validation : son gain est donc "
    "surestimé (le hasard du découpage a aidé), et il devrait en partie fondre sur un autre "
    "découpage."
)
print("\n[HYPOTHÈSE DE SÉLECTION D'OPTIMISME]\n", optimisme)

meilleur = max(scores, key=scores.get)

# Calculate out-of-sample variation delta on seed=1 to verify optimization drift
gain2 = (score_sur(meilleur, 1) - score_sur("base", 1)) / erreur_type
verifie_par_second_tirage = gain2 

print(f"\n-> Meilleur modèle sélectionné : {meilleur}")
print(f"-> Gain vérifié sur le second découpage (barres de bruit, graine 1) : {verifie_par_second_tirage:.2f}")


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural dataframe validation check
assert mes_essais.shape == (3, 6), "Erreur : La structure matricielle de la table des résultats est incorrecte."

# 2. Pipeline transformation integrity check (Data Leakage / Contamination Guard)
assert X_sans_features["age"].isna().sum() == 0, "Erreur : La variable globale source X_sans_features a été corrompue."

# 3. Model outputs bounds alignment
for s in scores.values():
    assert 0.0 <= s <= 100.0, f"Erreur : Score d'exactitude ({s:.2f}%) hors de l'espace mathématique de définition."

# 4. Consistency verify for seed outputs extraction
score_base_graine1 = score_sur("base", 1)
assert 0.0 <= score_base_graine1 <= 100.0, "Erreur lors de la récupération du score de référence sur le second découpage."

print("\n[✓] Tous les contrôles structurels et de validation stochastique hors-échantillon ont réussi.")

### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
# TODO : l'écart des deux familles, divisé par la barre de bruit.
# rapport = … (à vous)
# TODO : trois coefficients COMPARABLES, lus sur des colonnes de même nature ou remises à
#        la même échelle ; nommez-les.
# trois_coefficients = … (à vous)
# pourquoi_pas_les_dupliquees = … (à vous)
# quartier_provoque = … (à vous : cause, ou autre chose ?)
# cout_et_rapport = … (à vous : ce qu il coûte, ce qu il rapporte)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


In [ ]:
import pandas as pd
import numpy as np

# --- 0. Data Setup (Assuming variables from prior pipeline steps) ---
# score_interne = ... (Tree-based model score)
# score_lineaire = ... (Linear model score)
# erreur_type = ...
# modele_lineaire = ... (Fitted LogisticRegression / Linear model)
# Xtr_n = ... (Encoded training features dataframe)

# --- 1. Score Discrepancy (in standard error units) ---
rapport = (score_interne - score_lineaire) / erreur_type 
print("écart arbres/linéaire en barres :", rapport) 

# --- 2. Extract Top 3 Specific Indicators Coefficients ---
# Extracted .coef_[0] assuming binary classification setup
coefs = pd.Series(modele_lineaire.coef_[0], index=Xtr_n.columns) 

# Isolate indicators matching specific categorical prefixes
indic = coefs[[c for c in coefs.index if c.startswith(("quartier_", "assurance_"))]] 
trois_coefficients = list(indic.abs().sort_values(ascending=False).index[:3]) 

print("\nTop 3 coefficients pour quartier_ / assurance_ :")
print(coefs[trois_coefficients])  # Comparable because they share the same binary scale

# --- 3. Model Interpretation Texts ---
pourquoi_pas_les_dupliquees = ( 
    "age et age_mois (et montant_acte et cout_acte) portent la même information : le modèle " 
    "répartit le poids entre elles de façon arbitraire, donc chaque coefficient pris seul n'a " 
    "plus de sens, et il dépend aussi de l'unité." 
) 

quartier_provoque = ( 
    "Non : un coefficient décrit une association sur des données observées, pas une cause. "
    "Autre lecture : le quartier peut refléter la distance, le revenu ou l'accès aux transports " 
    "(variable cachée), pas le quartier lui-même." 
) 

cout_et_rapport = ( 
    "Coût : indicatrices (beaucoup de colonnes), comblement des valeurs manquantes (fillna) et " 
    "mise à l'échelle à faire. Rapport : un coefficient signé par colonne, lisible, que " 
    "l'on peut expliquer à un non-spécialiste, pour un écart de score de " 
    f"{rapport:.1f} barre(s)." 
) 

print("\n[INTERPRÉTATION : DOUBLONS DE FEATURES]")
print(pourquoi_pas_les_dupliquees)
print("\n[INTERPRÉTATION : CAUSALITÉ VS ASSOCIATION]")
print(quartier_provoque)
print("\n[INTERPRÉTATION : RAPPORT COÛT/BÉNÉFICE]")
print(cout_et_rapport)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural dimensions check on the linear model output weights
assert len(coefs) == Xtr_n.shape[1], \
    f"Error: Coefficient vector length ({len(coefs)}) does not match training features columns count ({Xtr_n.shape[1]})."

# 2. Validation that target features are present in the dataframe context
assert len(indic) > 0, \
    "Error: No columns starting with 'quartier_' or 'assurance_' were found in Xtr_n."

# 3. Validation of top 3 ranking selection limits
assert len(trois_coefficients) == 3 or len(trois_coefficients) == len(indic), \
    f"Error: Expected to select up to 3 elements, found {len(trois_coefficients)}."

# 4. Check that coefficients are ordered properly by absolute weight values
if len(trois_coefficients) >= 2:
    assert abs(coefs[trois_coefficients[0]]) >= abs(coefs[trois_coefficients[1]]), \
        "Error: The extracted coefficients are not properly sorted by absolute magnitude."

print("\n[✓] All feature coefficients alignments and interpretative matrix validation checks passed successfully.")

### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
# TODO : votre feature, dans `ajouter_features` (mise en place) — PAS `delai_long`.
# et ici : le constat qui la motive, le gain, le contournement.
# constat_d_origine = … (à vous : l'observation du constat de la question 3)
# gain_de_ma_feature = … (à vous : en barres)
# score_contourne = … (à vous)
# ecart_du_contournement = … (à vous : en barres)
# quelle_etape_oubliee = … (à vous)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


In [ ]:
import numpy as np
import pandas as pd

# --- (a) Feature Engineering Definition ---
def ajouter_features(df):
    """Adds missing indicator flags and replaces sentinel values with standard NaNs."""
    df = df.copy()
    df["age_manquant"] = (df["age"] < 0).astype(int)  # Binary indicator flag
    df["age_corrige"] = df["age"].where(df["age"] >= 0)  # Replaces -1 with NaN
    return df

# --- (b) Feature Impact Measurement ---
constat_d_origine = "un âge impossible (valeur -1) dans age"  # From Q3
prevoir("feature âge corrigé", "~0 ou léger +", "l'arbre gérait déjà le -1 : la table est sur un plateau")

# Assuming remesurer(), score_interne, and erreur_type are defined in your prior environment
score_avec_feature = remesurer()
gain_de_ma_feature = (score_avec_feature - score_interne) / erreur_type
print(f"Score avec feature: {score_avec_feature:.2f}% | Gain: {gain_de_ma_feature:.2f} barres")

# --- (c) Manual Reconstruction with Omitted Categorical Imputation ---
brut = train.loc[Xval.index]
columns_to_drop = [c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns]
d = brut.drop(columns=columns_to_drop).copy()

for c in CAT:
    # Omit fillna("") to intentionally recreate the silent missing value category bug
    d[c] = d[c].astype("category").cat.set_categories(X[c].cat.categories)

d = ajouter_features(d)

# Evaluate predictions from the model object
score_contourne = 100 * (modele.predict(d) == yval.values).mean()
ecart_du_contournement = (score_contourne - score_avec_feature) / erreur_type

quelle_etape_oubliee = (
    "Le comblement des valeurs manquantes des colonnes catégorielles par la chaîne vide "
    "(fillna('')) : sans lui, les NaN ne tombent pas dans la même catégorie qu'à "
    "l'entraînement. C'est un bug silencieux : aucune erreur, le score bouge seulement."
)

print(f"Score contourné: {score_contourne:.2f}% | Écart: {ecart_du_contournement:.2f} barres")
print("\n[EXPLICATION DU BUG SILENCIEUX]\n", quelle_etape_oubliee)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural dimensions integrity check
assert "age_manquant" in d.columns and "age_corrige" in d.columns, \
    "Error: Feature engineering columns were not appended to the reconstructed dataframe."

# 2. Functional check on data mutations inside the local scope
assert d["age_manquant"].isin([0, 1]).all(), \
    "Error: The missing age indicator contains non-binary types."
assert not (d["age"] < 0).any() == False or d.loc[d["age"] < 0, "age_corrige"].isna().all(), \
    "Error: Not all negative values in the source column were successfully masked into NaNs."

# 3. Check for the silent bug signature (Validating categorical NaN leaks)
# If the original raw validation slice contained missing strings, they must now be unmapped NaNs in the bug version
original_nans = brut[CAT].isna().sum().sum()
if original_nans > 0:
    assert d[CAT].isna().sum().sum() >= original_nans, \
        "Error: Categorical NaNs were unintentionally filled or omitted from the test target structure."

# 4. Range check for final validation percentages
for s in [score_avec_feature, score_contourne]:
    assert 0.0 <= s <= 100.0, f"Error: Calculated classification accuracy ({s:.2f}%) is outside bounds."

print("\n[✓] All structural bug assertions and feature injection checks passed successfully.")

### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
# TODO : l'écart de remplissage entre train et test, colonne par colonne.
# ecarts = … (à vous)
# colonne_trouvee = … (à vous)
# ecart_remplissage = … (à vous : en points)
# pourquoi_pas_la_derivee = … (à vous)
# remplissage_du_test = … (à vous : en pourcentage)
# gain_interne = … (à vous : en points)
# gain_en_barres = … (à vous)
# TODO : la simulation — la colonne manquante au taux du test, dans la validation.
# score_simule = … (à vous)
# chute_en_barres = … (à vous)
# decision_A_ECARTER = … (à vous)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.model_selection import train_test_split

# --- 1. Identify Covariate Shift / Missingness Discrepancies ---
communes = [c for c in train.columns if c in test.columns]
ecarts = ((train[communes].notna().mean() - test[communes].notna().mean()).abs() * 100).sort_values(ascending=False)

print("Top 5 des écarts de remplissage (Train vs Test) :")
print(ecarts.head(5))

colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(ecarts.iloc[0])
seconde = ecarts.index[1]

pourquoi_pas_la_derivee = (
    f"{seconde} (écart {ecarts.iloc[1]:.1f} points) est une dérive : elle reste renseignée "
    "dans le test, seulement un peu moins souvent. Une dérive n'est pas une disparition : "
    "on la surveille, on ne l'écarte pas pour cette raison."
)
print("\n[ANALYSE DU DEVIATION] ", pourquoi_pas_la_derivee)

remplissage_du_test = 100 * test[colonne_trouvee].notna().mean()

# --- 2. Evaluate Internal Feature Importance ---
col = colonne_trouvee

def table(df, avec):
    """Prepares feature space matrices by systematically dropping categories."""
    cols = [c for c in A_ECARTER if not (avec and c == col)] + [CIBLE] + TROP_FIN
    t = df.drop(columns=[c for c in cols if c in df.columns]).copy()
    
    for c in CAT:
        if c in t.columns:
            t[c] = t[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
            
    if avec and (col in t.columns) and not pd.api.types.is_numeric_dtype(t[col]):
        t[col] = t[col].astype("category")
        
    return t

ym = mon_train[CIBLE]
Xm_avec, Xm_sans = table(mon_train, True), table(mon_train, False)

a1, b1, c1, d1 = train_test_split(Xm_avec, ym, test_size=0.2, stratify=ym, random_state=0)
a0, b0, c0, d0 = train_test_split(Xm_sans, ym, test_size=0.2, stratify=ym, random_state=0)

m_avec = HGB(random_state=0).fit(a1, c1)
m_sans = HGB(random_state=0).fit(a0, c0)

s_avec_col = 100 * m_avec.score(b1, d1)
s_sans_col = 100 * m_sans.score(b0, d0)

# Local noise standard deviation calculation (in percentage points scale)
barre_m = 100 * ((s_avec_col / 100) * (1 - s_avec_col / 100) / len(d1)) ** 0.5
gain_interne = s_avec_col - s_sans_col
gain_en_barres = gain_interne / barre_m

# --- 3. Production Data Disappearance Simulation ---
taux_trous = 1 - (remplissage_du_test / 100)
rng = np.random.default_rng(0)
b_sim = b1.copy()

# Inject missing values at the test set failure rate
masque = rng.random(len(b_sim)) < taux_trous
b_sim.loc[masque, col] = np.nan

score_simule = 100 * m_avec.score(b_sim, d1)
chute_en_barres = (s_avec_col - score_simule) / barre_m

print("\n--- Résultats de l'impact Interne vs Dégradation ---")
print(f"Gain Interne : {gain_interne:.2f}% ({gain_en_barres:.1f} barres)")
print(f"Score Simulé (Dégradé Prod) : {score_simule:.2f}%")
print(f"Chute mesurée : {chute_en_barres:.1f} barres")

decision_A_ECARTER = (
    f"Oui, bonne décision : en production la colonne n'est renseignée que pour "
    f"{remplissage_du_test:.1f} % des lignes, et la simulation montre que le gain interne "
    f"({gain_en_barres:.1f} barres) s'évapore ({chute_en_barres:.1f} barres de chute). "
    "Mais la bonne raison n'est pas un score : c'est la disparition de la colonne au moment "
    "de décider ; un gain mesuré en interne ne dit rien de la production."
)
print("\n[VERDICT FINAL DU MODÈLE RÉSISTANT]")
print(decision_A_ECARTER)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural dimensions guard
assert col in Xm_avec.columns, f"Erreur : La colonne ciblée '{col}' doit être présente dans Xm_avec."
assert col not in Xm_sans.columns, f"Erreur : La colonne ciblée '{col}' doit être absente dans Xm_sans."

# 2. Validation of the simulation missing mask intensity bounds
assert 0.0 <= taux_trous <= 1.0, f"Erreur : Taux de corruption calculé ({taux_trous:.4f}) invalide."
if taux_trous > 0:
    assert b_sim[col].isna().sum() > b1[col].isna().sum(), \
        "Erreur : Le processus de simulation n'a injecté aucune nouvelle valeur manquante (NaN)."

# 3. Scale verification for the local standard deviation unit vector
assert barre_m > 0, "Erreur : La barre de bruit calculée doit être strictement positive pour éviter les divisions par zéro."

# 4. Check for score properties constraints
for s in [s_avec_col, s_sans_col, score_simule]:
    assert 0.0 <= s <= 100.0, f"Erreur : La précision calculée ({s:.2f}%) sort de la plage mathématique permise."

print("\n[✓] Tous les contrôles de dérive temporelle et de simulation stochastique ont réussi.")

### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'outil, avec les DEUX signes, qui ignore identifiant et cible.
def suspects(df, cible):
    """Rend une table, une ligne par colonne suspecte : nom, écart de remplissage, signe.
       Signe 1 (le VIDE) : plus de 50 points d'écart de remplissage entre les deux groupes.
       Signe 2 (la VALEUR) : une valeur qui ne vit que dans un seul des deux groupes.
       Ajoutez ce que vous jugez utile — mais jamais en lisant la cible par accident."""
    # … (à vous)

# TODO : l'outil appliqué à `train`.
# table_suspects = … (à vous)
# classement_suspects = … (à vous)
# TODO : le troisième signe — la sentinelle.
# sentinelle_trouvee = … (à vous)
# lignes_porteuses = … (à vous)
# taux_sur_ces_lignes = … (à vous : en pourcentage)
# verdict_sentinelle = … (à vous)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


In [ ]:
import pandas as pd
import numpy as np

# --- 1. Leakage Scanner Function Definition ---
def suspects(df, cible):
    """Generates a matrix checking for columns displaying high missingness imbalance or category leak signatures."""
    lignes = []
    g1 = df[cible] == 1
    
    for c in df.columns:
        if c in (cible, ID):
            continue
            
        r1 = df.loc[g1, c].notna().mean() * 100
        r0 = df.loc[~g1, c].notna().mean() * 100
        ecart = abs(r1 - r0)
        
        # Sign 1: High missingness pattern disparity (Leaking Empty vs Filled state)
        if ecart > 50:
            lignes.append({
                "colonne": c, 
                "ecart_remplissage": round(ecart, 1),
                "signe": "vide"
            })
            
        # Sign 2: Group exclusive category leak
        elif min(r1, r0) > 0 and df[c].nunique() <= 50:
            v1 = set(df.loc[g1, c].dropna().unique())
            v0 = set(df.loc[~g1, c].dropna().unique())
            excl = v1 ^ v0
            n_porteuses = df[c].isin(excl).sum()
            
            if excl and n_porteuses >= 30:
                lignes.append({
                    "colonne": c, 
                    "ecart_remplissage": round(ecart, 1),
                    "signe": f"valeur exclusive à un groupe : {sorted(map(str, excl))[:3]}"
                })
                
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage", "signe"])

# --- 2. Leakage Scanning and Metadata Verification ---
table_suspects = suspects(train, CIBLE)
display(table_suspects)

# Leakage Dictionary Framework 
CONNUS = {
    "motif_absence": ("fuite", "Le motif n'existe que si le patient est absent : il est connu après l'issue, c'est la réponse elle-même."),
    "duree_consultation": ("fuite", "Une durée de consultation n'existe que si la consultation a eu lieu, donc si le patient est venu."),
    "constantes_relevees": ("fuite", "Les constantes sont relevées pendant la consultation : connues après l'issue, pas au moment de décider."),
}

lignes = []
for c in table_suspects["colonne"]:
    verdict, just = CONNUS.get(c, ("à discuter", "À COMPLÉTER : dis ce que la valeur veut dire et quand elle est connue (avant ou après l'issue)."))
    lignes.append({"colonne": c, "verdict": verdict, "justification": just})

classement_suspects = pd.DataFrame(lignes)
display(classement_suspects)

# --- 3. Sign 3: Sentinel Values Deep Dive ---
neg = train["age"] < 0
valeur_sentinelle = train.loc[neg, "age"].unique()
sentinelle_trouvee = f"age = {', '.join(map(str, valeur_sentinelle))}"
lignes_porteuses = int(neg.sum())

taux_sur_ces_lignes = 100 * train.loc[neg, CIBLE].mean() if lignes_porteuses > 0 else 0.0
taux_global = 100 * train[CIBLE].mean()

print(f"\nSentinelle trouvée: {sentinelle_trouvee}")
print(f"Lignes porteuses: {lignes_porteuses} | Taux ciblé: {taux_sur_ces_lignes:.1f}% | Taux global: {taux_global:.1f}%")

verdict_sentinelle = (
    f"Défaut de saisie probable : age = -1 est un code mis à la place d'un âge manquant, porté "
    f"par {lignes_porteuses} ligne(s) seulement ; leur taux d'absence ({taux_sur_ces_lignes:.1f} %) "
    f"est à comparer au taux global ({taux_global:.1f} %). S'il est proche, ce n'est pas une "
    "fuite ; s'il est très éloigné sur peu de lignes, on le surveille (adapte à TES chiffres)."
)
print("\n[VERDICT SENTINELLE]", verdict_sentinelle)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural output consistency checks
assert isinstance(table_suspects, pd.DataFrame), "Erreur : suspects doit retourner un DataFrame pandas."
if not table_suspects.empty:
    assert list(table_suspects.columns) == ["colonne", "ecart_remplissage", "signe"], \
        "Erreur : Colonnes du DataFrame suspect non conformes."

# 2. Variable exclusion safety enforcement
assert CIBLE not in table_suspects["colonne"].values, "Erreur : La variable CIBLE ne peut pas s'auto-détecter suspecte."
if ID in train.columns:
    assert ID not in table_suspects["colonne"].values, "Erreur : L'identifiant (ID) a pollué le scan d'anomalies."

# 3. Sentinel calculation validation properties 
assert 0.0 <= taux_sur_ces_lignes <= 100.0, "Erreur : Le taux d'absence calculé sur les sentinelles est invalide."
assert 0.0 <= taux_global <= 100.0, "Erreur : Le taux global d'absence calculé est invalide."

print("\n[✓] Tous les contrôles de détection de fuite de données et d'intégrité de la table ont réussi.")

### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
# TODO : votre annonce, AVANT de regarder les tranches.
# annonce = … (à vous)
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
# ecart_global = … (à vous : en points)
# tranches = … (à vous : taux avec, taux sans, effectifs, écart, barre, écart en barres)
# tranches_exclues = … (à vous : celles qui ne portent pas 500 lignes, et pourquoi)
# delai_median_avec = … (à vous)
# delai_median_sans = … (à vous)
# variable_qui_explique = … (à vous)
# experience_qui_trancherait = … (à vous : trois lignes)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# --- 0. Columns Extraction ---
RAPPEL = [c for c in mon_train.columns if "rappel" in c.lower()][0]
DELAI = [c for c in mon_train.columns if "delai" in c.lower() and pd.api.types.is_numeric_dtype(mon_train[c])][0]
print("colonnes utilisées :", RAPPEL, DELAI)

# --- 1. Baseline Predictions & Assertions ---
# WRITTEN BEFORE LOOKING AT SLICES
annonce = (
    "J'attends que le rappel ne nuise pas ou aide un peu une fois le délai égalisé : l'écart "
    "global vient d'une autre variable (le délai). Je changerais d'avis si, dans chaque tranche "
    "de délai, le taux d'absence avec rappel reste supérieur de plus de 2 barres."
)
print("\n[ANNONCE INTUITIVE]\n", annonce)

avec = mon_train[RAPPEL].astype(str).str.lower().isin(["1", "1.0", "oui", "true", "yes"])
taux_avec = 100 * mon_train.loc[avec, CIBLE].mean()
taux_sans = 100 * mon_train.loc[~avec, CIBLE].mean()
ecart_global = taux_avec - taux_sans

# --- 2. Stratification Loop across Delay Intervals ---
bornes = [-1, 3, 7, 14, 30, 60, 90, 1e9]
tr = pd.cut(mon_train[DELAI], bornes)
tranches, tranches_exclues = [], []

for t in tr.cat.categories:
    m_t = tr == t
    n1, n0 = int((m_t & avec).sum()), int((m_t & ~avec).sum())
    
    if n1 < 500 or n0 < 500:
        tranches_exclues.append({
            "tranche": str(t), 
            "avec": n1, 
            "sans": n0, 
            "pourquoi": "un des deux groupes a moins de 500 lignes"
        })
        continue
        
    p1 = mon_train.loc[m_t & avec, CIBLE].mean()
    p0 = mon_train.loc[m_t & ~avec, CIBLE].mean()
    
    # Combined standard error bar computation for two independent proportions
    barre = 100 * (p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0) ** 0.5  
    ecart = 100 * (p1 - p0)
    
    tranches.append({
        "tranche": str(t), 
        "taux_avec": 100 * p1, 
        "taux_sans": 100 * p0, 
        "lignes": n1 + n0, 
        "avec": n1, 
        "sans": n0, 
        "ecart": ecart, 
        "barre": barre, 
        "ecart_en_barres": ecart / barre if barre > 0 else 0.0
    })

df_tranches = pd.DataFrame(tranches)
df_exclues = pd.DataFrame(tranches_exclues)

display(df_tranches)
display(df_exclues)

# --- 3. Operational Confounding Verdicts ---
delai_median_avec = float(mon_train.loc[avec, DELAI].median())
delai_median_sans = float(mon_train.loc[~avec, DELAI].median())

variable_qui_explique = (
    f"Le délai de prise de rendez-vous ({DELAI}) : médiane {delai_median_avec:.0f} jours avec "
    f"rappel contre {delai_median_sans:.0f} sans. Le rappel est surtout envoyé aux rendez-vous "
    "lointains, qui manquent plus souvent : (à confirmer par TES tranches)."
)

experience_qui_trancherait = (
    "Tirer AU HASARD, parmi les patients, ceux qui reçoivent un rappel et ceux qui n'en "
    "reçoivent pas. Comparer ensuite les taux d'absence des deux groupes. Le tirage au hasard "
    "rend les deux groupes semblables en délai et en tout le reste : la seule différence est "
    "le rappel, donc la comparaison est honnête."
)

print("\n[VERDICT VARIANCE CONVERTIE]", variable_qui_explique)
print("\n[EXPÉRIENCE IDÉALE (A/B TEST)]", experience_qui_trancherait)


# --- 4. Question 14 Framework Setup ---
# Mapping custom registration wrapper logic `hypothese()` if initialized by environment
if "hypothese" in globals():
    hypothese(
        "âge corrigé", 
        "age = -1 est un faux âge : le corriger (+ drapeau) aide un peu (signe +)", 
        "remesurer() avec la feature, gain en barres vs score_interne", 
        "un gain inférieur à 2 barres de bruit (|gain| < 2 barres)"
    )
    hypothese(
        "délai en semaines", 
        "un changement d'unité du délai n'apporte rien pour un arbre (signe 0)", 
        "ajouter delai / 7 et mesurer le gain en barres",
        "un gain inférieur à 2 barres de bruit (|gain| < 2 barres)"
    )


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Verification of baseline percentage score boundaries
assert 0.0 <= taux_avec <= 100.0 and 0.0 <= taux_sans <= 100.0, \
    "Erreur : Les taux moyens calculés sortent de l'espace mathématique [0, 100]."

# 2. Early-exit size verification on loop splits
if not df_tranches.empty:
    assert (df_tranches["avec"] >= 500).all() and (df_tranches["sans"] >= 500).all(), \
        "Erreur : Une tranche validée contient un sous-échantillon inférieur à 500 lignes."

# 3. Stratification allocation size verification 
total_rows_accounted = (df_tranches["lignes"].sum() if not df_tranches.empty else 0) + \
                      (df_exclues["avec"].sum() + df_exclues["sans"].sum() if not df_exclues.empty else 0)
assert total_rows_accounted == len(mon_train), \
    f"Erreur de découpage : Le total des lignes stratifiées ({total_rows_accounted}) diverge de mon_train ({len(mon_train)})."

# 4. Range validity verification for medians
assert delai_median_avec >= 0 and delai_median_sans >= 0, \
    "Erreur : Les délais médians extraits ne peuvent pas être négatifs."

print("\n[✓] Tous les contrôles de stratification par tranches et de détection de biais ont réussi.")

### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")
# TODO : une hypothèse TESTÉE, sous son propre nom.
# essai("ma feature", lambda: modele.score(Xval, yval))
# confirmee_ou_contredite = … (à vous : « confirmée » ou « contredite »)
# suite_donnee = … (à vous : ce que vous en faites)

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


In [ ]:
import numpy as np
import pandas as pd

# --- 0. Hypotheses Registration ---
# Assuming hypothese(nom, description, protocole, condition_rejet) is defined in your environment
if "hypothese" in globals():
    hypothese(
        "âge corrigé", 
        "age = -1 est un faux âge : le corriger (+ drapeau) aide un peu (signe +)", 
        "remesurer() avec la feature, gain en barres vs score_interne", 
        "un gain inférieur à 2 barres de bruit (|gain| < 2 barres)"
    )
    hypothese(
        "délai en semaines", 
        "un changement d'unité du délai n'apporte rien pour un arbre (signe 0)", 
        "ajouter delai / 7 et mesurer le gain en barres", 
        "un gain supérieur à 2 barres : les arbres seraient sensibles à l'échelle"
    )
    hypothese(
        "ratio cout_acte / montant_acte", 
        "les deux colonnes sont proportionnelles : le ratio est constant, inutile (signe 0)", 
        "vérifier que le ratio a un seul écart-type nul, puis mesurer le gain", 
        "un ratio qui varie d'une ligne à l'autre ou un gain supérieur à 2 barres"
    )

# --- 1. Experiment Execution & Prediction Tracking ---
# Test of ONE specific hypothesis matching your tracking metrics
if "prevoir" in globals():
    prevoir("âge corrigé", "~0 ou léger +", "critère annoncé : gain >= 2 barres sinon contredite")

# Run evaluation wrapper (ensuring fallback if custom tracking framework handles formatting)
if "essai" in globals():
    s14 = essai("âge corrigé", lambda: remesurer() / 100, changement="age -1 -> NaN + drapeau")
else:
    s14 = remesurer() / 100  # Fallback: fraction out of 1.0

# Bring accuracy to percentage scale (out of 100) if returned as a fraction
s14_pct = s14 * 100 if s14 <= 1.0 else s14
gain14 = (s14_pct - score_interne) / erreur_type
confirmee_ou_contredite = "confirmée" if gain14 >= 2 else "contredite"

print(f"Score mesuré : {s14_pct:.2f}% | Gain en barres : {gain14:.2f} | Hypothèse : {confirmee_ou_contredite}")

suite_donnee = (
    f"Gain de {gain14:.2f} barre(s) : l'hypothèse est {confirmee_ou_contredite}. Un verdict "
    f"« {confirmee_ou_contredite} » apprend que le modèle gérait déjà la sentinelle : je garde la feature "
    "ou non selon qu'elle simplifie la chaîne, et je cherche le signal ailleurs."
)
print("\n[VERDICT STRATÉGIQUE]\n", suite_donnee)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Structural check on score outputs bounds
assert 0.0 <= s14_pct <= 100.0, f"Erreur : Le score d'exactitude converti ({s14_pct:.2f}%) n'est pas un pourcentage valide."

# 2. Logic gate verification for binary hypothesis classification status
assert conf_status in ["confirmée", "contredite"], "Erreur : Statut de validation inattendu."
if gain14 < 2.0:
    assert conf_status == "contredite", "Erreur : L'hypothèse aurait dû être rejetée car sous le seuil de 2 barres de bruit."
else:
    assert conf_status == "confirmée", "Erreur : L'hypothèse aurait dû être acceptée car le gain atteint ou dépasse 2 barres."

# 3. Unit validation check for the noise factor variable
assert erreur_type > 0, "Erreur : L'écart-type de bruit (erreur_type) doit être strictement supérieur à zéro."

print("\n[✓] Tous les enregistrements d'hypothèses et contrôles stochastiques de l'étape 14 ont réussi.")
*

### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
# familles = … (à vous : {colonne: famille})
# TODO : une phrase de justification par colonne.
# justifications = … (à vous : {colonne: phrase})

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


In [ ]:
import pandas as pd

# --- 1. Family Categorization Mapping ---
familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "utilisable",
    "score_interne": "utilisable",
}

# --- 2. Professional Justifications Mapping ---
justifications = {
    "id_client": "Il désigne un client, il ne décrit pas la demande : il ne dit rien du risque.",
    "date_defaut": "Elle n'est renseignée que si le défaut a eu lieu : c'est l'issue elle-même.",
    "statut_6_mois": "Relevé six mois après l'octroi : il n'existe pas encore au moment de décider.",
    "montant": "Le montant est connu dès la demande et décrit la ligne : disponible à la décision.",
    "canal": "Le canal (agence, web, téléphone) est connu à la demande ; à surveiller s'il cesse d'être alimenté.",
    "score_interne": "Calculé au moment de la demande : disponible à la décision, c'est la définition.",
}

# Optional visual confirmation
df_audit = pd.DataFrame({
    "Colonne": familles.keys(),
    "Statut": familles.values(),
    "Justification": [justifications[k] for k in familles.keys()]
})
display(df_audit)


# --- CONTRÔLES ET ASSERTIONS ---

# 1. Bi-directional key alignment mapping check
assert familles.keys() == justifications.keys(), \
    "Erreur : Incohérence de clés entre le dictionnaire des familles et celui des justifications."

# 2. Guard against data leakage statuses missing justification triggers
for col, statut in familles.items():
    if statut == "connue après l'issue":
        assert "après" in justifications[col] or "issue" in justifications[col], \
            f"Erreur d'audit : La justification pour '{col}' doit explicitement mentionner le problème de temporalité."

# 3. Categorical boundary integrity check
valid_statuses = {"identifiant", "connue après l'issue", "utilisable"}
assert set(familles.values()).issubset(valid_statuses), \
    f"Erreur : Un statut non répertorié a été détecté dans les valeurs : {set(familles.values()) - valid_statuses}"

print("\n[✓] Tous les contrôles d'audit de variables et de cohérence de nomenclature ont réussi.")

## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
